In [1]:
#This is program for count VLA research

In [2]:
import pandas as pd
import arxiv

In [7]:
def set_client_search(query: str):
    client = arxiv.Client(
        page_size = 1000,   # 1リクエストでの抽出数。2000以下, default 100
        delay_seconds = 3.0,  # 遅延時間（秒）。rxivの規約上、1リクエスト以上/3秒はNG
    )

    query = ('abs:"autonomous driving"''AND (abs:"VLA" OR abs:"vision language model") ')

    client = arxiv.Client(
        page_size = 100,      # 修正: 1000→100。1リクエストあたりの負荷を下げる
        delay_seconds = 3.0,
        num_retries = 5,      # 修正: 503を踏んだ際のリトライ回数を増やす（default 3）
    )
    search = arxiv.Search(
        query = query,
        max_results = 1000,
        sort_by = arxiv.SortCriterion.SubmittedDate,   # 修正: relevance→投稿日
        sort_order = arxiv.SortOrder.Descending,
    )
    return client, search

# 要約にRAGを含み、カテゴリ: AI、期間; 20210101-20250504で検索
query = ('("abs:"autonomous driving" )' 'AND("abs:"VLA"OR abs:"Vision language model" )''AND (submittedDate:[202101010000 TO 202607252359])' )
client, search = set_client_search(query)
results = client.results(search)

In [39]:
def convert_to_str(vals):  # 文字列に変換
    if isinstance(vals, list):
        return list(map(str, vals))
    elif vals is None:
        return vals
    else:
        return str(vals)
        
def create_df(results):
    res0 = []
    for res in results:
        res_dict = dict(
            entry_id = res.entry_id,
            pdf_url = res.pdf_url,
            updated = res.updated, 
            published = res.published, 
            title = res.title, 
            authors = convert_to_str(res.authors),  # 文字列に変換
            summary = res.summary,
            comment = res.comment,
            journal_ref = res.journal_ref,
            doi = res.doi,
            primary_category = res.primary_category,
            categories = res.categories,
            links = convert_to_str(res.links),  # 文字列に変換
        )
        res0.append(res_dict)
    df = pd.DataFrame(res0)
    ndata = len(df)
    return df

df = create_df(results)


In [41]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET

NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def arxiv_total(query: str) -> int:
    url = ('http://export.arxiv.org/api/query?search_query='
           + urllib.parse.quote(query) + '&max_results=1')
    with urllib.request.urlopen(url) as r:
        root = ET.fromstring(r.read())
    return int(root.find('opensearch:totalResults', NS).text)

print(arxiv_total(query))

429


In [23]:
import time

tests = [
    '(abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]',
    'abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]',
    'cat:cs.RO AND abs:"autonomous driving" AND abs:"vision language model"',
]
for q in tests:
    print(arxiv_total(q), '|', q)
    time.sleep(3)

12275 | (abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]
336 | abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]
123 | cat:cs.RO AND abs:"autonomous driving" AND abs:"vision language model"


In [58]:
tests = [
    '(abs:"VLA" OR abs:"vision language model") AND submittedDate:[202101010000 TO 202607252359]',
    'abs:"autonomous driving" AND abs:"vision language model" AND submittedDate:[202101010000 TO 202607252359]',
]

In [45]:
import numpy as np

def create_url(aid: str):
    apath = aid.split("/")[-1].split("v")[0]
    arxiv_id = f'ARXIV:{apath}'
    search_url = f'{semantic_base_url}/{arxiv_id}' + '?fields=' + fields 
    return arxiv_id

semantic_base_url = 'https://api.semanticscholar.org/graph/v1/paper'  # Semantic ScholarのベースURL
fields = 'citationCount,influentialCitationCount,referenceCount'  # サイトから取得する追加項
df['arxiv_id'] = df['entry_id'].apply(create_url)
tcol = fields.split(",") 
df[tcol] = np.nan  # 初期化

In [46]:
import requests
import time

HEADERS = {}   # APIキー取得後は {"x-api-key": API_KEY}

def post_batch(batch, max_retry=6):
    for attempt in range(max_retry):
        r = requests.post(semantic_base_url + '/batch',
                          params={'fields': fields}, json={"ids": batch},
                          headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = int(r.headers.get('Retry-After', 0)) or 5 * 2 ** attempt
            print(f'  {r.status_code}: retry in {wait}s (attempt {attempt+1})')
            time.sleep(wait)
            continue
        r.raise_for_status()
    return None


def get_citation_batch(df, start=0, batch_size=500):
    outs = []
    data = df['arxiv_id'].tolist()
    for i in range(start, len(data), batch_size):
        batch = data[i:i+batch_size]
        payload = post_batch(batch)
        if payload is None:
            print('Error: giving up. restart with start=', i)
            break
        rows = {pid: rec for pid, rec in zip(batch, payload) if rec is not None}
        if rows:
            outs.append(pd.DataFrame.from_dict(rows, orient="index"))
        time.sleep(5)
    if not outs:
        return df
    res = pd.concat(outs).rename_axis('arxiv_id').reset_index()
    return df.merge(res[['arxiv_id'] + tcol], on='arxiv_id', how='left')


start = 0  # 開始行
batch_size = 500  # バッチ数。500以下
df = get_citation_batch(df, start=start, batch_size=batch_size)

In [47]:
df = df.drop(columns=[c + '_x' for c in tcol])
df = df.rename(columns={c + '_y': c for c in tcol})

In [50]:
pd.set_option("display.max_colwidth", None)

In [51]:
df.sort_values(['citationCount'],ascending=False).head(15)

,entry_id,pdf_url,updated,published,title,authors,summary,comment,journal_ref,doi,primary_category,categories,links,arxiv_id,citationCount,influentialCitationCount,referenceCount
119,http://arxiv.org/abs/2506.13757v3,https://arxiv.org/pdf/2506.13757v3,2025-11-05 23:46:20+00:00,2025-06-16 17:58:50+00:00,AutoVLA: A Vision-Language-Action Model for End-to-End Autonomous Driving with Adaptive Reasoning and Reinforcement Fine-Tuning,"[Zewei Zhou, Tianhui Cai, Seth Z. Zhao, Yun Zhang, Zhiyu Huang, Bolei Zhou, Jiaqi Ma]","Recent advancements in Vision-Language-Action (VLA) models have shown promise for end-to-end autonomous driving by leveraging world knowledge and reasoning capabilities. However, current VLA models often struggle with physically infeasible action outputs, complex model structures, or unnecessarily long reasoning. In this paper, we propose AutoVLA, a novel VLA model that unifies reasoning and action generation within a single autoregressive generation model for end-to-end autonomous driving. AutoVLA performs semantic reasoning and trajectory planning directly from raw visual inputs and language instructions. We tokenize continuous trajectories into discrete, feasible actions, enabling direct integration into the language model. For training, we employ supervised fine-tuning to equip the model with dual thinking modes: fast thinking (trajectory-only) and slow thinking (enhanced with chain-of-thought reasoning). To further enhance planning performance and efficiency, we introduce a reinforcement fine-tuning method based on Group Relative Policy Optimization (GRPO), reducing unnecessary reasoning in straightforward scenarios. Extensive experiments across real-world and simulated datasets and benchmarks, including nuPlan, nuScenes, Waymo, and CARLA, demonstrate the competitive performance of AutoVLA in both open-loop and closed-loop settings. Qualitative results showcase the adaptive reasoning and accurate planning capabilities of AutoVLA in diverse scenarios.",NeurIPS 2025; Website link:https://autovla.github.io/,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2506.13757v3, https://arxiv.org/pdf/2506.13757v3]",ARXIV:2506.13757,222,27,104
123,http://arxiv.org/abs/2505.17685v3,https://arxiv.org/pdf/2505.17685v3,2025-11-11 01:31:25+00:00,2025-05-23 09:55:32+00:00,FutureSightDrive: Thinking Visually with Spatio-Temporal CoT for Autonomous Driving,"[Shuang Zeng, Xinyuan Chang, Mengwei Xie, Xinran Liu, Yifan Bai, Zheng Pan, Mu Xu, Xing Wei, Ning Guo]","Vision-Language-Action (VLA) models offer significant potential for end-to-end driving, yet their reasoning is often constrained by textual Chains-of-Thought (CoT). This symbolic compression of visual information creates a modality gap between perception and planning by blurring spatio-temporal relations and discarding fine-grained cues. We introduce FSDrive, a framework that empowers VLAs to ""think visually"" using a novel visual spatio-temporal CoT. FSDrive first operates as a world model, generating a unified future frame that combines a predicted background with explicit, physically-plausible priors like future lane dividers and 3D object boxes. This imagined scene serves as the visual spatio-temporal CoT, capturing both spatial structure and temporal evolution in a single representation. The same VLA then functions as an inverse-dynamics model to plan trajectories conditioned on current observations and this visual CoT. We enable this with a unified pre-training paradigm that expands the model's vocabulary with visual tokens and jointly optimizes for semantic understanding (VQA) and future-frame prediction. A progressive curriculum first generates structural priors to enforce physical laws before rendering the full scene. Evaluations on nuScenes and NAVSIM show FSDrive improves trajectory accuracy and reduces collisions, while also achieving competitive FID for video generation with a lightweight autoregressive model and advancing scene understanding on DriveLM. These resul

In [19]:
import time, requests, urllib.parse

base = 'https://export.arxiv.org/api/query?search_query='
tail = '&start=0&max_results=5&sortBy=submittedDate&sortOrder=descending'

qs = [
    'all:electron',                                          # 対照。単純
    'abs:"autonomous driving"',                              # フレーズ1個
    'cat:cs.RO AND abs:"autonomous driving"',                # cat先行で絞る
    'abs:"autonomous driving" AND abs:"vision language model"',   # フレーズ2個
    'abs:"autonomous driving" AND (abs:"VLA" OR abs:"vision language model")',  # 本命
]
for q in qs:
    try:
        r = requests.get(base + urllib.parse.quote(q) + tail, timeout=60)
        print(r.status_code, len(r.content), '|', q)
    except Exception as e:
        print('EXC', type(e).__name__, '|', q)
    time.sleep(10)

EXC ReadTimeout | all:electron
200 13085 | abs:"autonomous driving"
200 12716 | cat:cs.RO AND abs:"autonomous driving"
200 14317 | abs:"autonomous driving" AND abs:"vision language model"
200 13787 | abs:"autonomous driving" AND (abs:"VLA" OR abs:"vision language model")


In [44]:
client = arxiv.Client(
    page_size = 100,
    delay_seconds = 3.0,
    num_retries = 10,      # 間欠障害が前提なので回数を確保する
)
search = arxiv.Search(
    query = 'abs:"autonomous driving" AND (abs:"VLA" OR abs:"vision language action")',
    max_results = 1000,
    sort_by = arxiv.SortCriterion.SubmittedDate,
    sort_order = arxiv.SortOrder.Descending,
)
df = create_df(client.results(search))
print(len(df))

INFO:arxiv:Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=abs%3A%22autonomous+driving%22+AND+%28abs%3A%22VLA%22+OR+abs%3A%22vision+language+action%22%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=100
INFO:arxiv:Got first page: 100 of 131 total results
INFO:arxiv:Sleeping: 2.968035 seconds
INFO:arxiv:Requesting page (first: False, try: 0): https://export.arxiv.org/api/query?search_query=abs%3A%22autonomous+driving%22+AND+%28abs%3A%22VLA%22+OR+abs%3A%22vision+language+action%22%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=100&max_results=100


131


In [17]:
import time, requests, urllib.request, urllib.error, urllib.parse

q = 'abs:"autonomous driving" AND (abs:"VLA" OR abs:"vision language model")'
enc = urllib.parse.quote(q)
tail = '&start=0&max_results=5&sortBy=submittedDate&sortOrder=descending'
http_  = 'http://export.arxiv.org/api/query?search_query='
https_ = 'https://export.arxiv.org/api/query?search_query='

def via_urllib(url):
    try:
        with urllib.request.urlopen(url) as r:
            return r.status, len(r.read())
    except urllib.error.HTTPError as e:
        return e.code, None

def via_requests(url, ua=None):
    r = requests.get(url, headers=({'User-Agent': ua} if ua else {}), timeout=30)
    return r.status_code, len(r.content)

tests = [
    ('urllib   / http  / 複雑', lambda: via_urllib(http_ + enc + tail)),
    ('urllib   / https / 複雑', lambda: via_urllib(https_ + enc + tail)),
    ('requests / https / 単純', lambda: via_requests(https_ + 'all:electron&max_results=1')),
    ('requests / https / 複雑', lambda: via_requests(https_ + enc + tail)),
    ('requests / https / UA変更', lambda: via_requests(https_ + enc + tail, 'arxiv-research/1.0')),
]
for name, fn in tests:
    print(name, fn())
    time.sleep(10)

urllib   / http  / 複雑 (429, None)
urllib   / https / 複雑 (503, None)
requests / https / 単純 (200, 2938)


ReadTimeout: HTTPSConnectionPool(host='export.arxiv.org', port=443): Read timed out. (read timeout=30)

In [43]:
NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}
for y in range(2015, 2027):
    q = f'abs:"autonomous driving" AND submittedDate:[{y}01010000 TO {y}12312359]'
    q = f'abs:"autonomous driving" AND (abs:"VLA" OR abs:"vision language action") AND submittedDate:[{y}01010000 TO {y}12312359]'

    print(y, arxiv_total(q))
    time.sleep(3)

2015 0
2016 0
2017 0
2018 0
2019 0
2020 0
2021 0
2022 0
2023 0
2024 4
2025 45
2026 82
